In [1]:
import cv2
import einops
import matplotlib.pyplot as plt
import mediapy
import numpy as np

import os

from openpi.policies.libero_reason_dataset import LiberoSkillReasonDataset
from openpi.training import config as _config

In [2]:
data_config = _config.get_config('pi05_libero_skill_reason_fixed')
dataset = LiberoSkillReasonDataset(data_config.data.base_config, data_config.model.action_horizon)

The dataset you requested (None) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=None
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).



Resolving data files:   0%|          | 0/4338 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/163 [00:00<?, ?it/s]

Using new skill reasoning dataset (Apr. 14)
self.prob_predict_prev_skills--->  0.0 0.5


In [42]:
import einops
import flax.nnx as nnx
import jax

class DenoiseNetwork(nnx.Module):
    def __init__(self, rngs):
        self.in_proj = nnx.Linear(in_features=16*6+1, out_features=2048, rngs=rngs)
        self.inner = nnx.Linear(in_features=2048, out_features=2048, rngs=rngs)
        self.inner2 = nnx.Linear(in_features=2048, out_features=2048, rngs=rngs)
        self.out_proj = nnx.Linear(in_features=2048, out_features=16*6, rngs=rngs)

    def __call__(self, x):
        # x = einops.rearrange(x, "b l d -> b (l d)")
        x = self.in_proj(x)
        x = jax.nn.relu(x)
        x = self.inner(x)
        x = self.inner2(x)
        x = jax.nn.relu(x)
        x = self.out_proj(x)
        return x


In [16]:
import tqdm
import random
n_samples = 10000
all_trajectories = np.zeros((n_samples, 16*7))
indices = random.sample(range(len(dataset)), n_samples)
for i in tqdm.tqdm(range(n_samples)):
    all_trajectories[i] = dataset[indices[i]]['actions'].cpu().numpy().flatten()

100%|██████████████████████████████████| 10000/10000 [09:45<00:00, 17.09it/s]


In [17]:
np.save("sample_trajectories.npy", all_trajectories)

In [66]:
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

import jax
#jax.config.update("jax_platforms", "gpu")
import jax.numpy as jnp

import flax.nnx as nnx
import optax

model = DenoiseNetwork(nnx.Rngs(0))
model.train()

schedule = optax.warmup_cosine_decay_schedule(
  init_value=0.0,
  peak_value=1e-4,
  warmup_steps=50,
  decay_steps=20000,
  end_value=5e-6,
)
optimizer = nnx.Optimizer(model, optax.adam(learning_rate=schedule), wrt=nnx.Param)

@nnx.jit
def train_step(model, optimizer, trajectory, rng):
    def compute_loss(model, rng):
        noise_rng, time_rng = jax.random.split(rng)
        # Copy pi05 diffusion parameters
        time = jax.random.beta(time_rng, 1.5, 1, len(trajectory)) * 0.999 + 0.001
        time = time[..., None]
        noise = jax.random.normal(rng, trajectory.shape)
        noised_trajectory = time * noise + (1-time) * trajectory
        predictions = model(jnp.concatenate([trajectory, time], axis=-1))
        return jnp.mean((predictions - trajectory)**2)

    loss, grads = nnx.value_and_grad(compute_loss)(model, rng)
    optimizer.update(grads)  # nnx.jit allows in place updates
    return loss

key = jax.random.key(0)
trajectories = jnp.load("sample_trajectories.npy")[:, :16*6]
batch_size = 200
i = 0
while i < 200000:
    key, perm_key, noise_key = jax.random.split(key, num=3)
    indices = jax.random.permutation(perm_key, len(trajectories))

    for j in range(0, len(trajectories), batch_size):
        trajectory_batch = trajectories[indices[j:j+batch_size]]
        #print(intermediates_batch.shape)
        #print(targets_batch.shape)
        loss = train_step(model, optimizer, trajectory_batch, perm_key)
        i += 1
        if i % 1000 == 0:
            print(f'step {i}')
            print(f'{loss = }')

step 1000
loss = Array(0.00029491, dtype=float32)
step 2000
loss = Array(0.00014783, dtype=float32)
step 3000
loss = Array(0.00011082, dtype=float32)
step 4000
loss = Array(8.562675e-05, dtype=float32)
step 5000
loss = Array(6.814906e-05, dtype=float32)
step 6000
loss = Array(5.9633305e-05, dtype=float32)
step 7000
loss = Array(4.0703428e-05, dtype=float32)
step 8000
loss = Array(3.7014735e-05, dtype=float32)
step 9000
loss = Array(3.0680745e-05, dtype=float32)
step 10000
loss = Array(2.524948e-05, dtype=float32)
step 11000
loss = Array(2.1815238e-05, dtype=float32)
step 12000
loss = Array(1.7180524e-05, dtype=float32)
step 13000
loss = Array(1.481019e-05, dtype=float32)
step 14000
loss = Array(1.3393789e-05, dtype=float32)
step 15000
loss = Array(1.1827175e-05, dtype=float32)
step 16000
loss = Array(1.0245785e-05, dtype=float32)
step 17000
loss = Array(8.91537e-06, dtype=float32)
step 18000
loss = Array(8.8926e-06, dtype=float32)
step 19000
loss = Array(8.465838e-06, dtype=float32)
st

In [68]:
import orbax.checkpoint as ocp
_, state = nnx.split(model)
checkpointer = ocp.StandardCheckpointer()
ckpt_dir = ocp.test_utils.erase_and_create_empty(os.path.abspath('generator'))
checkpointer.save(ckpt_dir / 'state', state)

In [69]:
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from pathlib import Path

def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_depths": True}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

class LiberoEnvMaker:
    def __init__(self, suite: str,
                 render_resolution: int = 256, seed: int = 0,
                 repeats: int = 1):
        benchmark_dict = benchmark.get_benchmark_dict()
        self.task_suite = benchmark_dict[suite]()
        self.repeats = repeats
        self.render_resolution = render_resolution
        self.seed = seed

    def get_num_tasks(self):
        return self.task_suite.n_tasks

    def task_instantiations(self, task_id):
        task = self.task_suite.get_task(task_id)
        initial_states = self.task_suite.get_task_init_states(task_id)
        env, task_description = _get_libero_env(task, self.render_resolution, self.seed)
        for episode_idx in range(self.repeats):
            env.reset()
            obs = env.set_init_state(initial_states[episode_idx])
            yield obs, env, task_description

In [70]:
N_REPEATS = 9999
libero_90 = LiberoEnvMaker("libero_90", repeats=N_REPEATS)
libero_10 = LiberoEnvMaker("libero_10", repeats=N_REPEATS)
task_generators = [libero_90.task_instantiations(i) for i in range(libero_90.get_num_tasks())] \
                + [libero_10.task_instantiations(i) for i in range(libero_10.get_num_tasks())]

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [71]:
def get_action(previous_chunk, rng, t0=1):
    fill_size = len(previous_chunk)
    if fill_size == 0:
        filled_action = jax.random.normal(rng, (16, 6))
    else:
        filled_action = jnp.concat([previous_chunk, previous_chunk[-1] + t0*jax.random.normal(rng, (16-fill_size, 6))], axis=0)
    filled_action = filled_action.flatten()

    time = t0
    dt = 0.05
    action = filled_action
    while time > 0:
        inputs = jnp.concatenate([action, jnp.array([time])])
        # add and remove batch dimension
        action = model(inputs.reshape((1, *inputs.shape)))[0]
        time -= dt
    return action.reshape([16, 6])

print(get_action([], jax.random.key(0)))

[[-0.2872652  -0.7944256  -1.0600202  -0.26818475  0.28608295 -0.22933191]
 [-0.2127602   0.34563547 -0.80180275 -0.4835776  -0.19837227  0.3877109 ]
 [-0.15635423 -0.2957753   0.50549734 -0.42092887 -0.29273188  0.2548509 ]
 [-0.20069924 -0.3390516  -0.78788763 -0.81820494 -0.18958825 -0.27548137]
 [ 0.22258955 -0.29263324 -0.15008111 -0.4165548  -0.74282014 -0.3001922 ]
 [ 0.07214585  0.01889246 -0.04677014 -0.58515656 -0.6194129  -1.2266914 ]
 [-0.7681346   0.14484255  0.19759017  0.38574657 -0.27669442 -0.51667047]
 [-0.60928065  0.88286    -0.63443744  0.53392255 -0.6652203  -0.4218977 ]
 [-0.06504267 -1.1007783   0.5059459  -0.13843979  0.11890313 -0.1356879 ]
 [-0.31011257 -0.06863862 -0.8692876  -0.41316244  0.23231462 -0.17083594]
 [-0.4109444  -0.42565724 -0.1957984   0.4581918  -0.83266133  0.8864534 ]
 [-0.17511916 -0.25221595 -0.427082   -1.0201162  -0.06427053 -1.2178425 ]
 [-0.00202176 -0.0614493   0.00596315 -0.6568358  -0.01939339 -0.7184216 ]
 [-0.95392597 -0.11788036

In [72]:
import mediapy

obs, env, task_description = next(task_generators[0])
video_frames = []
zeroed = False
action_i = 0

rng = jax.random.key(1)
rng, action_rng = jax.random.split(rng)
actions = get_action([], action_rng)
for i in range(60):
    if action_i >= 2:
        rng, action_rng = jax.random.split(rng)
        actions = get_action(actions[action_i:], action_rng, t0=0.05)
        action_i = 0
    full_action = np.concatenate([actions[action_i], [-1]])
    action_i += 1
    obs, reward, done, info = env.step(full_action)
    video_frames.append(np.copy(obs['agentview_image'][::-1, ::-1, :]))
mediapy.write_video("gen.mp4", video_frames)